# 00 · Master Pipeline (Colab)

Pipeline completo em **uma única célula**: setup do repositório (clone ou `git pull --ff-only`), inspeção do ambiente, coleta/resolução da fila AIR/URL/HF (**Civitai + Hugging Face**) com **INPUT RESOLUTION SUMMARY** e aborto antes de tocar no dataset se alguma entrada falhar, download sequencial para staging temporário (cache HF temporário + limpeza de `.cache`) e publicação no Kaggle Dataset (CLI com fallback kagglehub).

**Substitui os antigos notebooks 01, 02, 03 e 04.**

Pré-requisitos (Secrets do Colab, com **Notebook access** habilitado):
- `CIVITAI_TOKEN`
- `KAGGLE_USERNAME`, `KAGGLE_KEY` e `KAGGLE_DATASET_NAME` (ex: `comfydocs`)
- `HF_TOKEN` (opcional — só para repos Hugging Face privados/gated)

Ordem de execução: 1) injetar os Secrets nas variáveis de ambiente; 2) validar a suíte de testes; 3) rodar o pipeline.

O script é interativo: ele pede os AIRs/URLs/HF (digite `done` para fechar a lista), as notas da versão e a confirmação de publicação. Se alguma entrada falhar na **resolução** ou no **download**, o pipeline imprime o relatório e encerra com erro **sem** publicar o dataset (publicação transacional: o lote só sobe se todos os downloads do lote tiverem sucesso).


## Checklist E2E (Colab) — não baixar arquivos gigantes desnecessariamente

Use arquivos **pequenos** primeiro (1 HF público + 1 HF privado + 1 Civitai). Confirme no log as fases:

1. `parse` → `resolution` → `metadata` → `classification`
2. `staging` + cache HF (`HF_HOME` / `HF_HUB_CACHE` / `HUGGINGFACE_HUB_CACHE` apontando para temp)
3. `download` → `validation` → `publish` (só se 100% dos downloads OK) → `cleanup`

Depois repita com **um** arquivo maior. Em falha de download espere `[DOWNLOAD FAILED]` e `Nenhuma alteração no dataset foi publicada.`

HF privado real (opt-in, sem token no notebook):

```bash
# Secrets/env: HF_TOKEN + HF_PRIVATE_TEST_REPO (+ opcional HF_PRIVATE_TEST_FILE)
!python /content/colab-pipeline/scripts/e2e_hf_private_check.py
```

Se credenciais/repo não estiverem disponíveis, o script imprime `HF private repository E2E: NOT RUN` (não finja validação).


In [ ]:
# ============================================================
# SECRETS -> VARIÁVEIS DE AMBIENTE (execute ANTES do pipeline)
# ============================================================
# O master_pipeline.py lê os.environ. Configure os Secrets no Colab
# (⚙️ -> Secrets) e habilite "Notebook access" para cada um:
#   CIVITAI_TOKEN        (obrigatório) token da Civitai
#   KAGGLE_USERNAME      (obrigatório) seu username Kaggle
#   KAGGLE_KEY           (obrigatório) API key Kaggle
#   KAGGLE_DATASET_NAME  (obrigatório) nome do dataset, ex: comfydocs
#   HF_TOKEN             (opcional)    só repos Hugging Face privados/gated

import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

SECRETS = (
    'CIVITAI_TOKEN',
    'KAGGLE_USERNAME',
    'KAGGLE_KEY',
    'KAGGLE_DATASET_NAME',
    'HF_TOKEN',
)
REQUIRED = ('CIVITAI_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY', 'KAGGLE_DATASET_NAME')

missing = []
for name in SECRETS:
    value = os.environ.get(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception as exc:  # SecretNotFoundError: não existe ou sem Notebook access
            print(f'[WARN] {name}: indisponível nos Secrets ({type(exc).__name__})')
    if value:
        os.environ[name] = value
        print(f'✅ {name}: configurado')
    else:
        missing.append(name)

if missing:
    print('[WARN] Secrets ausentes/vazios: ' + ', '.join(missing))
for name in REQUIRED:
    if not os.environ.get(name):
        raise RuntimeError(
            'Secret obrigatório ausente: ' + name + '. '
            'Configure em Colab Secrets (ícone de chave), habilite o Notebook access e re-execute esta célula.'
        )
print('✅ Secrets injetados nas variáveis de ambiente')


In [ ]:
# ============================================================
# TESTS — roda a suite de kaggle_dataset_manager
# ============================================================
# Execute esta célula ANTES do master pipeline para validar o código.
# Não requer secrets nem interação; pode rodar em qualquer runtime Colab.
# ============================================================

import subprocess
from pathlib import Path

REPO_DIR = Path('/content/colab-pipeline')
REPO_URL = 'https://github.com/automadevs/colab-pipeline.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

!python -m unittest discover -s /content/colab-pipeline/tests -v; echo "TESTS_EXIT=$?"


In [ ]:
# ============================================================
# COLAB TRANSFER - 00: MASTER PIPELINE (CIVITAI -> KAGGLE)
# ============================================================
# Única célula necessária. Substitui 01, 02, 03 e 04.
# Downloads são SEQUENCIAIS (um arquivo por vez) por restrição do Civitai.
# A fila aceita AIR (urn:air:...), URL Civitai e Hugging Face
# (hf:org/repo/arquivo.safetensors ou URL huggingface.co).
# Opcional: sobrescrever o dataset alvo com --dataset "owner/nome".
# ============================================================

import subprocess
from pathlib import Path

REPO_DIR = Path('/content/colab-pipeline')
REPO_URL = 'https://github.com/automadevs/colab-pipeline.git'

# Bootstrap mínimo: garante que o script exista na 1ª execução.
# O master_pipeline.py faz 'git pull --ff-only' internamente a cada run.
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

!python /content/colab-pipeline/scripts/master_pipeline.py; echo "PIPELINE_EXIT=$?"